# 🔴 SafeRoute India — Notebook 01: Explore Crime Data

This notebook explores the raw and cleaned crime datasets used to build risk scores.

**Before running:** download data with `python src/01_download_data.py` and clean with `python src/02_clean_crime.py`

**Contents:**
1. Load and inspect raw NCRB/Kaggle CSVs
2. Crime type breakdown and severity weighting
3. State/district distribution maps
4. Year-on-year trend analysis
5. Crime score normalisation validation
6. Geocoded crime points — spatial overview

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import folium
from folium.plugins import HeatMap
from pathlib import Path

from config import CRIME_SEVERITY, TARGET_CITIES
from src.utils import normalise_series

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'

DATA_RAW  = Path('../data/raw/crime')
DATA_PROC = Path('../data/processed')
print('Paths configured.')

## 1. Load Raw Crime CSVs

In [ ]:
# List available raw crime files
raw_files = list(DATA_RAW.glob('*.csv'))
print(f'Found {len(raw_files)} raw crime CSV files:')
for f in raw_files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<50} {size_kb:>8.1f} KB')

In [ ]:
# Load the primary dataset (adapt filename to what you downloaded)
primary = next((f for f in raw_files if 'crime' in f.name.lower()), raw_files[0] if raw_files else None)

if primary is None:
    print('No raw crime files found. Run: python src/01_download_data.py')
else:
    df_raw = pd.read_csv(primary)
    print(f'Loaded: {primary.name}')
    print(f'Shape:  {df_raw.shape}')
    print(f'\nColumns: {list(df_raw.columns)}')
    df_raw.head()

## 2. Crime Type Breakdown & Severity Weights

In [ ]:
# Show the severity weight assigned to each crime type (from config.py)
sev_df = pd.DataFrame(
    [(k, v) for k, v in CRIME_SEVERITY.items()],
    columns=['Crime Type', 'Severity Weight']
).sort_values('Severity Weight', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(sev_df)))
bars = ax.barh(sev_df['Crime Type'], sev_df['Severity Weight'], color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, sev_df['Severity Weight']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Severity Weight (higher = more dangerous for travellers)', fontsize=11)
ax.set_title('Crime Severity Weights Used in Risk Scoring', fontsize=13, fontweight='bold')
ax.set_xlim(0, 12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/crime_severity_weights.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Load Cleaned Crime Data

In [ ]:
clean_path = DATA_PROC / 'crime_clean.csv'
if not clean_path.exists():
    print('Run first: python src/02_clean_crime.py')
else:
    df = pd.read_csv(clean_path)
    print(f'Cleaned crime data: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'\nStates covered: {df["STATE"].nunique()}')
    print(f'Districts covered: {df["DISTRICT"].nunique()}')
    if 'YEAR' in df.columns:
        print(f'Year range: {df["YEAR"].min()} – {df["YEAR"].max()}')
    df.head()

## 4. Top 20 Highest-Crime Districts

In [ ]:
score_col = next((c for c in df.columns if 'NORM' in c.upper()), None)
if score_col:
    top20 = df.nlargest(20, score_col)[['STATE', 'DISTRICT', score_col]].reset_index(drop=True)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    labels = [f"{row['DISTRICT']}\n({row['STATE']})" for _, row in top20.iterrows()]
    colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(top20)))
    bars = ax.bar(range(len(top20)), top20[score_col], color=colors[::-1], edgecolor='white', linewidth=0.3)
    ax.set_xticks(range(len(top20)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Normalised Crime Score (0–1)', fontsize=11)
    ax.set_title('Top 20 Highest-Crime Districts in Dataset', fontsize=13, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig('../outputs/top20_crime_districts.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(top20.to_string(index=False))

## 5. Crime Score Distribution

In [ ]:
if score_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df[score_col].dropna(), bins=50, color='#ef4444', alpha=0.8, edgecolor='white', lw=0.3)
    axes[0].axvline(df[score_col].mean(), color='black', ls='--', lw=2, label=f'Mean={df[score_col].mean():.3f}')
    axes[0].axvline(df[score_col].median(), color='blue', ls=':', lw=2, label=f'Median={df[score_col].median():.3f}')
    axes[0].set_xlabel('Normalised Crime Score')
    axes[0].set_ylabel('Number of Districts')
    axes[0].set_title('Crime Score Distribution')
    axes[0].legend()
    
    # CDF
    sorted_scores = np.sort(df[score_col].dropna())
    cdf = np.arange(1, len(sorted_scores)+1) / len(sorted_scores)
    axes[1].plot(sorted_scores, cdf, color='#ef4444', lw=2)
    axes[1].axvline(0.5, color='gray', ls='--', lw=1, label='Score=0.5')
    axes[1].fill_between(sorted_scores, cdf, alpha=0.15, color='#ef4444')
    axes[1].set_xlabel('Normalised Crime Score')
    axes[1].set_ylabel('Cumulative Proportion of Districts')
    axes[1].set_title('CDF of Crime Scores')
    axes[1].legend()
    
    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    plt.suptitle('Crime Score Distribution Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/crime_score_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Geocoded Crime Points — Spatial Heatmap

In [ ]:
geocoded_path = DATA_PROC / 'crime_geocoded.csv'
if not geocoded_path.exists():
    print('Run first: python src/05_geocode.py')
else:
    df_geo = pd.read_csv(geocoded_path).dropna(subset=['LAT','LON'])
    score_col_geo = next((c for c in df_geo.columns if 'NORM' in c.upper()), None)
    
    m = folium.Map(location=[20.5, 78.9], zoom_start=5, tiles='CartoDB dark_matter')
    
    heat_data = df_geo[['LAT','LON', score_col_geo]].values.tolist() if score_col_geo else df_geo[['LAT','LON']].assign(w=0.5).values.tolist()
    HeatMap(heat_data, radius=14, blur=10,
            gradient={0.2:'blue',0.45:'green',0.65:'yellow',0.85:'orange',1.0:'red'}).add_to(m)
    
    # City center markers
    for city_key, cfg in TARGET_CITIES.items():
        folium.CircleMarker(
            location=list(cfg['center']),
            radius=8, color='white', fill=True, fill_color='white',
            popup=city_key.title()
        ).add_to(m)
    
    out = '../outputs/crime_heatmap_india.html'
    m.save(out)
    print(f'Saved: {out}')
    print(f'Points plotted: {len(df_geo):,}')
    m

## 7. Per-City Crime Score Summary

In [ ]:
if 'df_geo' in dir() and score_col_geo:
    print('Crime score summary for target cities:\n')
    print(f'{"City":<15} {"State":<20} {"Points":>8} {"Mean Score":>12} {"Max Score":>10}')
    print('─' * 70)
    for city_key, cfg in TARGET_CITIES.items():
        bbox = cfg['bbox']  # (south, west, north, east)
        mask = (
            df_geo['LAT'].between(bbox[0], bbox[2]) &
            df_geo['LON'].between(bbox[1], bbox[3])
        )
        city_df = df_geo[mask]
        if len(city_df) == 0:
            print(f'{city_key:<15} {cfg["state"]:<20} {"No data":>8}')
        else:
            mean_s = city_df[score_col_geo].mean()
            max_s  = city_df[score_col_geo].max()
            print(f'{city_key:<15} {cfg["state"]:<20} {len(city_df):>8,} {mean_s:>12.3f} {max_s:>10.3f}')